# Fine-Tune Llama-3.2-1B-Instruct on Banking77

**Target:** Google Colab T4 GPU (16 GB VRAM)  
**Stack:** Unsloth + QLoRA (4-bit) + TRL SFTTrainer  
**Dataset:** `mteb/banking77` — 13,069 samples, 77 intent classes  
**Expected time:** ~45 minutes on T4

Run cells top-to-bottom. Checkpoints are saved to Google Drive every 30 minutes automatically.

## 0 — Runtime check
Verify we have a GPU before spending time on installs.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU found. Go to Runtime > Change runtime type > T4 GPU and re-run."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")

## 1 — Install dependencies

In [ ]:
# Unsloth install command from their official Colab instructions.
# This installs the CUDA-enabled build matched to the current torch version.
!pip install unsloth --quiet
!pip install trl peft datasets transformers accelerate bitsandbytes tensorboard wandb --quiet

print("Installation complete.")

## 2 — Mount Google Drive (for persistent checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All outputs go inside your Drive so they survive a session disconnect.
DRIVE_ROOT       = "/content/drive/MyDrive/banking77_finetune"
FINAL_MODEL_DIR  = f"{DRIVE_ROOT}/models/final_lora"
HOURLY_CKPT_DIR  = f"{DRIVE_ROOT}/checkpoints/periodic_backup"
CHECKPOINT_DIR   = f"{DRIVE_ROOT}/checkpoints"
LOG_DIR          = f"{DRIVE_ROOT}/logs"
DATA_PATH        = "/content/processed_banking77.jsonl"  # local, fast I/O

for d in [FINAL_MODEL_DIR, HOURLY_CKPT_DIR, CHECKPOINT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted. Output root:", DRIVE_ROOT)

## Configuration — set flags before running all cells

| Flag | Default | Meaning |
|---|---|---|
| `RESUME_MODE` | `False` | Set `True` to resume from a Drive checkpoint after a disconnect |
| `RUN_SWEEP` | `False` | Set `True` to run the W&B hyperparameter sweep after training |

**W&B API key**: add `WANDB_API_KEY` to Colab Secrets (🔑 icon, left sidebar) for fully hands-free login.

In [ ]:
# ── Run-mode flags — set these before hitting Runtime → Run all ───────────────

RESUME_MODE = False
# True  → Section 9 loads the latest Drive checkpoint instead of training
#          from scratch. Use this after a Colab session disconnect.
# False → Normal training run (Section 9 is skipped automatically).

RUN_SWEEP = False
# True  → Section 10 runs a W&B Bayesian hyperparameter sweep (~75 min extra).
# False → Single training run only (Section 10 is skipped automatically).

print(f"RESUME_MODE = {RESUME_MODE}")
print(f"RUN_SWEEP   = {RUN_SWEEP}")

## 3 — Prepare the Banking77 dataset

In [ ]:
import json
import logging
from collections import Counter
from datasets import load_dataset

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("prepare_data")

# 77 intent label names (index = label integer from the dataset)
LABEL_NAMES = [
    "activate_my_card", "age_limit", "apple_pay_or_google_pay", "atm_support",
    "automatic_top_up", "balance_not_updated_after_bank_transfer",
    "balance_not_updated_after_cheque_or_cash_deposit", "beneficiary_not_allowed",
    "cancel_transfer", "card_about_to_expire", "card_acceptance", "card_arrival",
    "card_delivery_estimate", "card_linking", "card_not_working", "card_payment_fee_charged",
    "card_payment_not_recognised", "card_payment_wrong_exchange_rate", "card_swallowed",
    "cash_withdrawal_charge", "cash_withdrawal_not_recognised", "change_pin",
    "compromised_card", "contactless_not_working", "country_support",
    "declined_card_payment", "declined_cash_withdrawal", "declined_transfer",
    "direct_debit_payment_not_recognised", "disposable_card_limits",
    "edit_personal_details", "exchange_charge", "exchange_rate", "exchange_via_app",
    "extra_charge_on_statement", "failed_transfer", "fiat_currency_support",
    "get_disposable_virtual_card", "get_physical_card", "getting_spare_card",
    "getting_virtual_card", "lost_or_stolen_card", "lost_or_stolen_phone",
    "order_physical_card", "passcode_forgotten", "pending_card_payment",
    "pending_cash_withdrawal", "pending_top_up", "pending_transfer",
    "pin_blocked", "receiving_money", "refund_not_showing_up",
    "request_refund", "reverted_card_payment?", "supported_cards_and_currencies",
    "terminate_account", "top_up_by_bank_transfer_charge", "top_up_by_card_charge",
    "top_up_by_cash_or_cheque", "top_up_failed", "top_up_limits",
    "top_up_reverted", "topping_up_by_card", "transaction_charged_twice",
    "transfer_fee_charged", "transfer_into_account", "transfer_not_received_by_recipient",
    "transfer_timing", "unable_to_verify_identity", "verify_my_identity",
    "verify_source_of_funds", "verify_top_up", "virtual_card_not_working",
    "visa_or_mastercard", "why_is_my_card_blocked", "wrong_amount_of_cash_received",
    "wrong_exchange_rate_for_cash_withdrawal",
]

INSTRUCTION = (
    "Classify the following banking customer query into one of 77 intent categories. "
    "Respond with only the intent label name."
)

def to_alpaca(sample):
    return {
        "instruction": INSTRUCTION,
        "input": sample["text"],
        "output": LABEL_NAMES[sample["label"]],
    }

logger.info("Loading mteb/banking77...")
raw = load_dataset("mteb/banking77")

# Keep train and test splits separate for evaluation
train_records = [to_alpaca(s) for s in raw["train"]]
eval_records  = [to_alpaca(s) for s in raw["test"]]

logger.info("Train split: %d samples", len(train_records))
logger.info("Eval split : %d samples", len(eval_records))

# Write only training records to JSONL (eval stays in memory for the callback)
with open(DATA_PATH, "w") as f:
    for rec in train_records:
        f.write(json.dumps(rec) + "\n")

logger.info("Saved %d training records to %s", len(train_records), DATA_PATH)
logger.info("Unique intents: %d", len({r['output'] for r in train_records}))
print("\nSample record:", train_records[0])

In [ ]:
import wandb

# Try Colab Secrets first (add WANDB_API_KEY via the 🔑 icon in the left sidebar).
# Falls back to an interactive prompt if the secret is not set.
try:
    from google.colab import userdata
    wandb.login(key=userdata.get("WANDB_API_KEY"))
    print("W&B login: using Colab Secret.")
except Exception:
    wandb.login()   # interactive fallback — will prompt for API key
    print("W&B login: interactive.")

## 4 — Load model with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME     = "unsloth/Llama-3.2-1B-Instruct"
MAX_SEQ_LENGTH = 512   # T4 has 16 GB — can afford longer sequences
LORA_RANK      = 16
LORA_ALPHA     = 32    # alpha = 2 * rank — more stable gradient scaling

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,   # QLoRA: base weights in 4-bit, adapters in bf16
    dtype=None,          # auto-detect (bf16 on T4)
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    use_rslora=True,     # rank-stabilized LoRA: more stable gradients at higher ranks
    random_state=42,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model.print_trainable_parameters()

## 5 — 30-minute checkpoint callback

In [ ]:
import random
import time
import wandb
from transformers import TrainerCallback

CHECKPOINT_INTERVAL_SECONDS = 1800  # 30 minutes


class TimeBasedCheckpointCallback(TrainerCallback):
    """
    Saves a timestamped checkpoint to Google Drive every 30 minutes of
    wall-clock training time. Keeps your work safe if Colab disconnects.
    """

    def __init__(self, save_dir: str, interval_seconds: float = CHECKPOINT_INTERVAL_SECONDS):
        self.save_dir         = save_dir
        self.interval_seconds = interval_seconds
        self._last_save_time  = time.time()
        self._save_count      = 0
        self._trainer         = None

    def on_step_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self._last_save_time
        if elapsed >= self.interval_seconds:
            self._save_count += 1
            ts        = time.strftime("%Y%m%d_%H%M%S")
            ckpt_path = os.path.join(self.save_dir, f"ckpt_{ts}")
            os.makedirs(ckpt_path, exist_ok=True)
            if self._trainer:
                self._trainer.save_model(ckpt_path)
                print(f"[Checkpoint #{self._save_count}] Saved to Drive: {ckpt_path}")
            self._last_save_time = time.time()
            control.should_save = False

    def attach_trainer(self, trainer):
        self._trainer = trainer


class GenerationEvalCallback(TrainerCallback):
    """
    After each epoch, runs generation on a 500-sample eval subset and logs
    exact-match accuracy to console + W&B.
    Uses the model's native chat template for prompt construction.
    """

    def __init__(self, eval_records, tokenizer, sample_size=500, max_new_tokens=20):
        random.seed(42)
        self.eval_sample    = random.sample(eval_records, min(sample_size, len(eval_records)))
        self.tokenizer      = tokenizer
        self.max_new_tokens = max_new_tokens

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        model.eval()
        correct = 0
        for rec in self.eval_sample:
            messages = [{"role": "user", "content": f"{rec['instruction']}\n\n{rec['input']}"}]
            prompt = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = self.tokenizer(
                prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH
            ).to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            pred = self.tokenizer.decode(
                out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            ).strip()
            if pred == rec["output"]:
                correct += 1
        acc = correct / len(self.eval_sample)
        print(f"\n[GenerationEval] Epoch {int(state.epoch)}: Accuracy = {acc:.4f} ({correct}/{len(self.eval_sample)})")
        if wandb.run:
            wandb.log({"eval/exact_match_accuracy": acc, "epoch": state.epoch})
        model.train()


print("Callbacks defined. Checkpoint interval:", CHECKPOINT_INTERVAL_SECONDS // 60, "min")

## 6 — Format dataset & train

In [ ]:
from datasets import load_dataset as hf_load_dataset, Dataset
from trl import SFTTrainer, SFTConfig


def format_prompt(sample):
    """Format using the model's native chat template instead of Alpaca."""
    messages = [
        {"role": "user",      "content": f"{sample['instruction']}\n\n{sample['input']}"},
        {"role": "assistant", "content": sample["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


# Training dataset — loaded from JSONL (fast local I/O)
raw_ds   = hf_load_dataset("json", data_files=DATA_PATH, split="train")
train_ds = raw_ds.map(format_prompt, remove_columns=raw_ds.column_names)

# Eval dataset — built from in-memory eval_records (no file I/O needed)
eval_raw = Dataset.from_list(eval_records)
eval_ds  = eval_raw.map(format_prompt, remove_columns=eval_raw.column_names)

print(f"Train dataset: {len(train_ds)} samples")
print(f"Eval dataset : {len(eval_ds)} samples")
print("Sample (chat template):\n", train_ds[0]["text"][:400])

In [ ]:
# Training config — tuned for T4 16 GB
wandb.init(project="banking77-finetune")

sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,    # effective batch = 16
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_dir=LOG_DIR,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",               # bitsandbytes 8-bit Adam — saves VRAM
    neftune_noise_alpha=5,            # NEFTune: embedding noise improves generation quality
    report_to=["tensorboard", "wandb"],
    dataloader_num_workers=2,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=True,                     # pack short sequences for faster training
    seed=42,
)

ckpt_callback = TimeBasedCheckpointCallback(save_dir=HOURLY_CKPT_DIR)
eval_callback = GenerationEvalCallback(eval_records, tokenizer)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    callbacks=[ckpt_callback, eval_callback],
)

ckpt_callback.attach_trainer(trainer)

trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in trainer.model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# --- TRAIN ---
try:
    result = trainer.train()
    print("Training complete:", result.metrics)
except KeyboardInterrupt:
    print("Interrupted — saving current state...")
finally:
    print(f"Saving final adapter to: {FINAL_MODEL_DIR}")
    trainer.save_model(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)
    print("Saved.")

## 7 — TensorBoard (optional, view training curves)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

## 8 — Inference test

In [ ]:
from unsloth import FastLanguageModel

TEST_QUERIES = [
    "I lost my card, what should I do?",
    "Can I use my card abroad?",
    "How do I change my PIN?",
    "My transfer hasn't arrived yet.",
    "What are the limits for topping up?",
]

# Enable Unsloth inference optimizations
FastLanguageModel.for_inference(model)

print(f"{'Query':<45} {'Predicted Intent':<30}")
print("-" * 75)

for query in TEST_QUERIES:
    messages = [{"role": "user", "content": f"{INSTRUCTION}\n\n{query}"}]
    prompt   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs   = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_ids    = outputs[0][inputs["input_ids"].shape[1]:]
    prediction = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    q_display  = (query[:42] + "...") if len(query) > 45 else query
    print(f"{q_display:<45} {prediction:<30}")

## 9 — Resume from checkpoint (if session was disconnected)

Run this cell **instead of cells 4–6** if your Colab session was killed mid-training.

In [ ]:
if RESUME_MODE:
    import os
    from pathlib import Path
    from unsloth import FastLanguageModel

    def find_latest_checkpoint(backup_dir: str) -> str | None:
        """Return the most recent timestamped checkpoint dir, or None."""
        p = Path(backup_dir)
        if not p.exists():
            return None
        candidates = sorted(
            [d for d in p.iterdir() if d.is_dir() and (d / "adapter_config.json").exists()],
            key=lambda d: d.name,
            reverse=True,
        )
        return str(candidates[0]) if candidates else None

    ckpt      = find_latest_checkpoint(HOURLY_CKPT_DIR)
    load_from = ckpt if ckpt else "unsloth/Llama-3.2-1B-Instruct"
    print("Loading from:", load_from)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=load_from,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,
    )
    print("Resumed. Re-run Section 6 onwards to continue training.")
else:
    print("RESUME_MODE = False — skipping. Set RESUME_MODE = True after a disconnect.")

## 10 — Hyperparameter Sweep with W&B

Run **after** completing a full training run. Uses Bayesian optimization to search over LoRA rank, learning rate, batch size, and alpha.

Each run trains for 1 epoch (~15 min on T4), so 5 runs ≈ 75 min total. Results are tracked in the `banking77-sweep` W&B project.

In [ ]:
if RUN_SWEEP:
    SWEEP_CONFIG = {
        "method": "bayes",
        "metric": {"name": "eval/exact_match_accuracy", "goal": "maximize"},
        "parameters": {
            "learning_rate": {"values": [1e-4, 2e-4, 5e-4]},
            "lora_rank":     {"values": [8, 16, 32]},
            # lora_alpha is derived as 2*rank inside train_sweep — not a free parameter
            "batch_size":    {"values": [4, 8]},
        },
        "early_terminate": {"type": "hyperband", "min_iter": 1},
    }
    sweep_id = wandb.sweep(SWEEP_CONFIG, project="banking77-sweep")
    print("Sweep ID:", sweep_id)
else:
    print("RUN_SWEEP = False — skipping sweep config.")

In [ ]:
if RUN_SWEEP:
    def train_sweep():
        with wandb.init() as run:
            cfg = wandb.config

            # Load a fresh model for each sweep run
            sweep_model, sweep_tokenizer = FastLanguageModel.from_pretrained(
                model_name=MODEL_NAME,
                max_seq_length=MAX_SEQ_LENGTH,
                load_in_4bit=True,
                dtype=None,
            )
            sweep_model = FastLanguageModel.get_peft_model(
                sweep_model,
                r=cfg.lora_rank,
                lora_alpha=cfg.lora_rank * 2,  # alpha = 2 * rank
                lora_dropout=0.05,
                bias="none",
                use_gradient_checkpointing="unsloth",
                use_rslora=True,
                random_state=42,
                target_modules=[
                    "q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",
                ],
            )

            sweep_sft_config = SFTConfig(
                output_dir="/content/sweep_tmp",
                num_train_epochs=1,    # 1 epoch per run keeps each run ~15 min
                per_device_train_batch_size=cfg.batch_size,
                gradient_accumulation_steps=max(1, 16 // cfg.batch_size),
                learning_rate=cfg.learning_rate,
                weight_decay=0.01,
                warmup_ratio=0.03,
                lr_scheduler_type="cosine",
                logging_steps=10,
                save_strategy="no",
                eval_strategy="epoch",
                fp16=not torch.cuda.is_bf16_supported(),
                bf16=torch.cuda.is_bf16_supported(),
                optim="adamw_8bit",
                neftune_noise_alpha=5,
                report_to=["wandb"],
                dataloader_num_workers=2,
                max_length=MAX_SEQ_LENGTH,
                dataset_text_field="text",
                packing=True,
                seed=42,
            )

            eval_cb = GenerationEvalCallback(eval_records, sweep_tokenizer)
            sweep_trainer = SFTTrainer(
                model=sweep_model,
                args=sweep_sft_config,
                train_dataset=train_ds,
                eval_dataset=eval_ds,
                processing_class=sweep_tokenizer,
                callbacks=[eval_cb],
            )
            sweep_trainer.train()

            del sweep_model, sweep_trainer
            torch.cuda.empty_cache()

    print("train_sweep() defined.")
else:
    print("RUN_SWEEP = False — skipping sweep function.")

In [ ]:
if RUN_SWEEP:
    # count=5 → 5 hyperparameter configurations, ~75 min total on T4
    wandb.agent(sweep_id, function=train_sweep, count=5)
else:
    print("RUN_SWEEP = False — skipping sweep agent.")

## 11 — Full Evaluation (all 3,076 eval samples)

Runs batched inference on the complete test split and reports:
- **Overall exact-match accuracy** on all 3,076 samples
- **Per-class accuracy** — worst and best 10 intents
- **Top confused pairs** — which intents the model mixes up most
- All results logged to W&B as searchable tables

Cell order:
1. Helper function definition (`run_full_eval`)
2. Run eval on fine-tuned model (~5–8 min)
3. *(Optional)* Base model zero-shot comparison (~25 min extra)

In [ ]:
from tqdm.auto import tqdm


def run_full_eval(model, tokenizer, eval_records, batch_size=32, max_new_tokens=20, desc="Evaluating"):
    """
    Batched generation eval over eval_records.
    Returns (preds, labels, accuracy).
    Left-pads batches so all sequences in a batch share the same length.
    """
    model.eval()
    tokenizer.padding_side = "left"

    all_preds  = []
    all_labels = [r["output"] for r in eval_records]

    for i in tqdm(range(0, len(eval_records), batch_size), desc=desc):
        batch   = eval_records[i : i + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": f"{r['instruction']}\n\n{r['input']}"}],
                tokenize=False, add_generation_prompt=True,
            )
            for r in batch
        ]
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_SEQ_LENGTH,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[1]   # same for all (left-padded)
        for out in outputs:
            pred = tokenizer.decode(out[prompt_len:], skip_special_tokens=True).strip()
            all_preds.append(pred)

    correct = sum(p == l for p, l in zip(all_preds, all_labels))
    return all_preds, all_labels, correct / len(all_labels)


print("run_full_eval() defined.")

In [ ]:
from collections import Counter

# ── Full evaluation on all 3,076 eval samples (~5–8 min on T4) ────────────────
FastLanguageModel.for_inference(model)

print("Running full evaluation on all 3,076 eval samples (batched, ~5–8 min)...\n")
ft_preds, ft_labels, ft_acc = run_full_eval(
    model, tokenizer, eval_records, batch_size=32, desc="Fine-tuned model"
)

print(f"\n{'='*62}")
print(f"  Fine-tuned model accuracy: {ft_acc:.4f}  ({int(ft_acc * len(ft_labels))}/{len(ft_labels)})")
print(f"{'='*62}")

# ── Per-class accuracy ─────────────────────────────────────────────────────────
per_class = {}
for pred, label in zip(ft_preds, ft_labels):
    if label not in per_class:
        per_class[label] = {"correct": 0, "total": 0}
    per_class[label]["total"]   += 1
    per_class[label]["correct"] += int(pred == label)

rows = sorted(
    [{"intent": k, "acc": v["correct"] / v["total"],
      "correct": v["correct"], "total": v["total"]}
     for k, v in per_class.items()],
    key=lambda x: x["acc"]
)

print("\n── Worst 10 intents ──────────────────────────────────────────")
for r in rows[:10]:
    bar = "█" * int(r["acc"] * 20)
    print(f"  {r['intent']:<45}  {r['acc']:.2f}  {r['correct']:>2}/{r['total']}  {bar}")

print("\n── Best 10 intents ───────────────────────────────────────────")
for r in rows[-10:][::-1]:
    bar = "█" * int(r["acc"] * 20)
    print(f"  {r['intent']:<45}  {r['acc']:.2f}  {r['correct']:>2}/{r['total']}  {bar}")

# ── Top confused intent pairs ──────────────────────────────────────────────────
confused = Counter(
    (label, pred)
    for pred, label in zip(ft_preds, ft_labels)
    if pred != label
)

print(f"\n── Top 15 confused pairs (true → predicted) {'─'*17}")
print(f"  {'True intent':<40}  {'Predicted as':<40}  Count")
print(f"  {'─'*40}  {'─'*40}  {'─'*5}")
for (true, pred), count in confused.most_common(15):
    print(f"  {true:<40}  {pred:<40}  {count:>5}")

# ── Log everything to W&B ─────────────────────────────────────────────────────
if wandb.run:
    wandb.log({"eval/full_exact_match_accuracy": ft_acc})

    class_table = wandb.Table(columns=["intent", "accuracy", "correct", "total"])
    for r in rows:
        class_table.add_data(r["intent"], r["acc"], r["correct"], r["total"])
    wandb.log({"eval/per_class_accuracy": class_table})

    confused_table = wandb.Table(columns=["true_intent", "predicted_as", "count"])
    for (true, pred), count in confused.most_common(30):
        confused_table.add_data(true, pred, count)
    wandb.log({"eval/confused_pairs": confused_table})

    print("\nFull results logged to W&B (tables viewable in the run dashboard).")

In [ ]:
# ── OPTIONAL: base model zero-shot comparison (~25 min on T4) ─────────────────
# Loads the original Llama-3.2-1B-Instruct without any LoRA to measure
# how much fine-tuning actually helped.

print("Loading base model (no LoRA) for zero-shot comparison...")
print("This takes ~25 min on T4. Skip this cell if you don't need the delta.\n")

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(base_model)

base_preds, _, base_acc = run_full_eval(
    base_model, base_tokenizer, eval_records,
    batch_size=32, desc="Base model (zero-shot)"
)

print(f"\n{'='*62}")
print(f"  Base model (zero-shot) : {base_acc:.4f}  ({int(base_acc * len(eval_records))}/{len(eval_records)})")
print(f"  Fine-tuned model       : {ft_acc:.4f}  ({int(ft_acc * len(eval_records))}/{len(eval_records)})")
print(f"  Improvement            : +{ft_acc - base_acc:.4f}  ({(ft_acc - base_acc)*100:.1f} pp)")
print(f"{'='*62}")

if wandb.run:
    wandb.log({
        "eval/base_exact_match_accuracy": base_acc,
        "eval/finetuning_improvement":    ft_acc - base_acc,
    })
    print("Logged to W&B.")

del base_model
torch.cuda.empty_cache()

In [ ]:
# ── 11b: Constrained decoding eval ────────────────────────────────────────────
# Builds a token trie over all 77 valid label names. At each generation step,
# only tokens that continue a valid label are allowed — the model physically
# cannot hallucinate an invalid label like "top_up_by_card".
#
# Expected delta vs unconstrained: small (~+0.1–0.5 pp).
# The invalid-label hallucinations are eliminated outright, but semantically
# confused pairs (e.g. topping_up_by_card → top_up_by_card_charge) are a
# weight problem, not a decoding problem — constrained decoding can't fix them.


def _build_trie(tokenizer, label_names):
    eos  = tokenizer.eos_token_id
    trie = {}
    for name in label_names:
        tokens = tokenizer.encode(name, add_special_tokens=False)
        node   = trie
        for tok in tokens:
            node = node.setdefault(tok, {})
        node[eos] = {}          # after a complete label, allow EOS to terminate
    return trie


def run_constrained_eval(model, tokenizer, eval_records, label_names,
                         batch_size=16, max_new_tokens=20, desc="Constrained eval"):
    """
    Trie-based prefix-constrained generation.
    At every decode step, only tokens that are valid continuations of some
    label name are allowed. Output is guaranteed to be in label_names.
    batch_size=16 (vs 32 unconstrained) — constraint logic adds overhead.
    """
    model.eval()
    tokenizer.padding_side = "left"
    eos  = tokenizer.eos_token_id
    trie = _build_trie(tokenizer, label_names)

    all_preds  = []
    all_labels = [r["output"] for r in eval_records]

    for i in tqdm(range(0, len(eval_records), batch_size), desc=desc):
        batch   = eval_records[i : i + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": f"{r['instruction']}\n\n{r['input']}"}],
                tokenize=False, add_generation_prompt=True,
            )
            for r in batch
        ]
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_SEQ_LENGTH,
        ).to(model.device)

        prompt_len = inputs["input_ids"].shape[1]

        def prefix_allowed_tokens_fn(batch_id, input_ids):
            generated = input_ids[prompt_len:].tolist()
            node = trie
            for tok in generated:
                if tok == eos:
                    return [eos]
                if tok in node:
                    node = node[tok]
                else:
                    # Generated token outside trie — open all tokens as fallback
                    return list(range(tokenizer.vocab_size))
            return list(node.keys()) if node else [eos]

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=eos,
                prefix_allowed_tokens_fn=prefix_allowed_tokens_fn,
            )

        for out in outputs:
            pred = tokenizer.decode(out[prompt_len:], skip_special_tokens=True).strip()
            all_preds.append(pred)

    correct = sum(p == l for p, l in zip(all_preds, all_labels))
    return all_preds, all_labels, correct / len(all_labels)


print("run_constrained_eval() defined.")

In [ ]:
print("Running constrained eval on all 3,076 samples (~8–12 min on T4)...\n")

con_preds, con_labels, con_acc = run_constrained_eval(
    model, tokenizer, eval_records, LABEL_NAMES,
    batch_size=16, desc="Constrained (trie)"
)

print(f"\n{'='*62}")
print(f"  Unconstrained (free generation) : {ft_acc:.4f}  ({int(ft_acc  * len(con_labels))}/{len(con_labels)})")
print(f"  Constrained   (trie decoding)   : {con_acc:.4f}  ({int(con_acc * len(con_labels))}/{len(con_labels)})")
delta = con_acc - ft_acc
print(f"  Delta                           : {delta:+.4f}  ({delta * 100:+.2f} pp)")
print(f"{'='*62}")

# ── What did constrained decoding actually change? ─────────────────────────────
fixed  = [(r["input"], r["output"], fp, cp)
          for r, fp, cp in zip(eval_records, ft_preds, con_preds)
          if fp != r["output"] and cp == r["output"]]
broken = [(r["input"], r["output"], fp, cp)
          for r, fp, cp in zip(eval_records, ft_preds, con_preds)
          if fp == r["output"] and cp != r["output"]]

print(f"\n  Samples fixed by constrained decoding  : {len(fixed)}")
print(f"  Samples broken by constrained decoding : {len(broken)}")

if fixed:
    print("\n── Fixed samples (sample) ────────────────────────────────────")
    for inp, true, fp, cp in fixed[:3]:
        print(f"  Query  : {inp[:65]}")
        print(f"  True   : {true}")
        print(f"  Before : {fp}  ← invalid or wrong")
        print(f"  After  : {cp}")
        print()

if broken:
    print("── Broken samples (sample) ───────────────────────────────────")
    for inp, true, fp, cp in broken[:3]:
        print(f"  Query  : {inp[:65]}")
        print(f"  True   : {true}")
        print(f"  Before : {fp}  ← was correct")
        print(f"  After  : {cp}  ← constrained pushed to wrong label")
        print()

if wandb.run:
    wandb.log({
        "eval/constrained_exact_match_accuracy":    con_acc,
        "eval/constrained_vs_unconstrained_delta":  delta,
        "eval/constrained_samples_fixed":           len(fixed),
        "eval/constrained_samples_broken":          len(broken),
    })
    print("Logged to W&B.")